# 02 — Power BI ML-Ready Table Preprocessing

This notebook prepares shared model inputs from the Power BI exported
ML-ready flat table.

It:

- standardizes column names;
- prepares the target;
- excludes surrogate keys and other non-feature columns;
- defines categorical and numeric features;
- creates the shared stratified train/test split;
- saves the split and its metadata.

Categorical encoding and numeric scaling are fitted inside each
model's Pipeline.

Existing train/test files are checked before saving. Matching split
files are preserved.

In [1]:
# Imports

from pathlib import Path
import json

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

In [2]:
# Paths

from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the project root whether the notebook is run from the notebook folder or the project root."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "_03_Models").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open the notebook from inside the project folder."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

ML_READY_DIR = PROJECT_ROOT / "data" / "3_ml_ready"
MODEL_INPUT_DIR = PROJECT_ROOT / "data" / "4_model_inputs"

ML_READY_PATH = ML_READY_DIR / "ML_StudentOutcome_Flat.csv"

MODEL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Expected ML-ready table:", ML_READY_PATH)
print("Model input output folder:", MODEL_INPUT_DIR)

Project root: D:\NU\semester_8\data_mining\project\phase I
Expected ML-ready table: D:\NU\semester_8\data_mining\project\phase I\data\3_ml_ready\ML_StudentOutcome_Flat.csv
Model input output folder: D:\NU\semester_8\data_mining\project\phase I\data\4_model_inputs


## 1. Load the Power BI exported ML-ready flat table

This file should be exported from Power BI after building the star schema and joining the fact table with the dimension tables.

Expected file:

```text
data/3_ml_ready/ML_StudentOutcome_Flat.csv
```

This table should contain one row per student record, all feature columns, and the target column.

In [3]:
if not ML_READY_PATH.exists():
    available_files = list(ML_READY_DIR.glob("*")) if ML_READY_DIR.exists() else []
    raise FileNotFoundError(
        "ML_StudentOutcome_Flat.csv was not found.\n"
        "First export the flat ML-ready table from Power BI to:\n"
        f"{ML_READY_PATH}\n\n"
        f"Available files in {ML_READY_DIR}: {available_files}"
    )

df = pd.read_csv(ML_READY_PATH, encoding="utf-8-sig")
print("Shape:", df.shape)
display(df.head())

Shape: (4424, 44)


,AcademicPerformanceKey,Admission grade,AdmissionKey,Age at enrollment,Application mode,Application order,Course,CourseKey,Curricular units 1st sem (approved),Curricular units 1st sem (credited),...,Mother's qualification,Nationality,Previous qualification,Previous qualification (grade),Scholarship holder,StudentKey,Target,TargetEncoded,Tuition fees up to date,Unemployment rate
0,1,100.0,853,33,39,1,171,1,0,0,...,37,1,1,133.1,1,2790,Dropout,0,1,15.5
1,1,100.0,2229,18,43,1,171,1,0,0,...,19,1,1,145.0,1,2598,Dropout,0,1,13.9
2,1,100.0,2826,27,39,1,171,1,0,0,...,12,1,1,110.0,0,3325,Dropout,0,1,9.4
3,1,100.8,2518,26,39,1,171,1,0,0,...,37,1,1,140.0,0,2954,Dropout,0,1,16.2
4,1,106.0,2084,36,39,1,171,1,0,0,...,37,1,12,110.0,0,2407,Graduate,2,1,7.6


## 2. Clean column names after export

Power BI exports may preserve spaces, tabs, or old column names.  
This step standardizes column names without changing the meaning of the dataset.

Important rule:

- `Application order` remains numeric/ordinal.
- Surrogate keys such as `StudentKey`, `CourseKey`, etc. are not model features.
- Target mapping remains:
  - Dropout → 0
  - Enrolled → 1
  - Graduate → 2

In [4]:
def clean_column_name(col: str) -> str:
    col = str(col).replace("\ufeff", "")
    col = col.replace("\t", " ")
    col = " ".join(col.split())
    col = col.replace("Nacionality", "Nationality")
    return col.strip()

df.columns = [clean_column_name(c) for c in df.columns]

# Standardize common target variants if needed.
target_name_variants = {
    "target": "Target",
    "Target ": "Target",
    "targetencoded": "TargetEncoded",
    "Target Encoded": "TargetEncoded",
    "Target_Encoded": "TargetEncoded",
}

renames = {}
for c in df.columns:
    normalized = c.replace(" ", "").replace("_", "").lower()
    if normalized == "target":
        renames[c] = "Target"
    elif normalized == "targetencoded":
        renames[c] = "TargetEncoded"

df = df.rename(columns=renames)

print(df.columns.tolist())

['AcademicPerformanceKey', 'Admission grade', 'AdmissionKey', 'Age at enrollment', 'Application mode', 'Application order', 'Course', 'CourseKey', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Daytime/evening attendance', 'Debtor', 'Displaced', 'EconomicContextKey', 'Educational special needs', "Father's occupation", "Father's qualification", 'FinancialKey', 'GDP', 'Gender', 'Inflation rate', 'International', 'Marital status', "Mother's occupation", "Mother's qualification", 'Nationality', 'Previous qualification', 'Previous qualification (grade)', 

In [5]:
# Basic validation

print("Shape:", df.shape)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", df.duplicated().sum())

if "Target" not in df.columns and "TargetEncoded" not in df.columns:
    raise ValueError("The flat ML-ready table must contain either 'Target' or 'TargetEncoded'.")

Shape: (4424, 44)

Missing values:


AcademicPerformanceKey                 0
Admission grade                        0
AdmissionKey                           0
Age at enrollment                      0
Application mode                       0
Application order                      0
Course                                 0
CourseKey                              0
Curricular units 1st sem (approved)    0
Curricular units 1st sem (credited)    0
dtype: int64


Duplicate rows: 0


## 3. Create target variable

If Power BI exported `TargetEncoded`, we use it directly.  
If not, we create it from the original `Target` column.

In [6]:
TARGET_MAPPING = {
    "Dropout": 0,
    "Enrolled": 1,
    "Graduate": 2,
}

TARGET_NAMES = {
    0: "Dropout",
    1: "Enrolled",
    2: "Graduate",
}

if "TargetEncoded" not in df.columns:
    df["TargetEncoded"] = df["Target"].map(TARGET_MAPPING)

if df["TargetEncoded"].isna().any():
    bad_values = df.loc[df["TargetEncoded"].isna(), "Target"].unique() if "Target" in df.columns else "Unknown"
    raise ValueError(f"Target encoding failed. Unexpected target values: {bad_values}")

df["TargetEncoded"] = df["TargetEncoded"].astype(int)

print("Target distribution:")
display(df["TargetEncoded"].map(TARGET_NAMES).value_counts())
display(df["TargetEncoded"].value_counts(normalize=True).rename("ratio"))

Target distribution:


TargetEncoded
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

TargetEncoded
2    0.499322
0    0.321203
1    0.179476
Name: ratio, dtype: float64

## 4. Select features and remove non-feature columns

The model should not train on DWH surrogate keys because they are artificial identifiers, not meaningful student attributes.

Dropped columns include:

- `Target`
- `TargetEncoded`
- any column ending with `Key`
- common index columns if exported accidentally

In [7]:
non_feature_columns = []

for col in df.columns:
    normalized = col.lower().replace(" ", "").replace("_", "")
    if col in ["Target", "TargetEncoded"]:
        non_feature_columns.append(col)
    elif col.endswith("Key"):
        non_feature_columns.append(col)
    elif normalized in ["index", "rownumber", "studentid", "id"]:
        non_feature_columns.append(col)

non_feature_columns = sorted(set(non_feature_columns))

X = df.drop(columns=non_feature_columns)
y = df["TargetEncoded"]

print("Dropped non-feature columns:")
print(non_feature_columns)

print("\nX shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

Dropped non-feature columns:
['AcademicPerformanceKey', 'AdmissionKey', 'CourseKey', 'EconomicContextKey', 'FinancialKey', 'StudentKey', 'Target', 'TargetEncoded']

X shape: (4424, 36)
y shape: (4424,)


,Admission grade,Age at enrollment,Application mode,Application order,Course,Curricular units 1st sem (approved),Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (grade),...,International,Marital status,Mother's occupation,Mother's qualification,Nationality,Previous qualification,Previous qualification (grade),Scholarship holder,Tuition fees up to date,Unemployment rate
0,100.0,33,39,1,171,0,0,0,0,0.0,...,0,1,9,37,1,1,133.1,1,1,15.5
1,100.0,18,43,1,171,0,0,0,0,0.0,...,0,1,9,19,1,1,145.0,1,1,13.9
2,100.0,27,39,1,171,0,0,0,0,0.0,...,0,1,5,12,1,1,110.0,0,1,9.4
3,100.8,26,39,1,171,0,0,0,0,0.0,...,0,1,9,37,1,1,140.0,0,1,16.2
4,106.0,36,39,1,171,0,0,0,0,0.0,...,0,2,7,37,1,12,110.0,0,1,7.6


## 5. Define categorical and numeric features

Some features are numeric-coded categories, so they should be one-hot encoded.

`Application order` is **not** one-hot encoded because it represents an ordinal ranking/order. It stays numeric and is scaled.

In [8]:
categorical_candidates = [
    "Marital status",
    "Application mode",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nationality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

categorical_features = [c for c in categorical_candidates if c in X.columns]
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Categorical/binary features:", len(categorical_features))
print(categorical_features)

print("\nNumeric/ordinal/count-based features:", len(numeric_features))
print(numeric_features)

if "Application order" in categorical_features:
    raise ValueError("Application order should be numeric/ordinal, not categorical.")

Categorical/binary features: 17
['Marital status', 'Application mode', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Nationality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']

Numeric/ordinal/count-based features: 19
['Admission grade', 'Age at enrollment', 'Application order', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'GDP', 'Inflation rate', 'Pr

## 6. Train/test split

We use one shared stratified split for all models.

This makes the model comparison fair because Logistic Regression, Naive Bayes, Decision Tree, Random Forest, and all later models will train and test on exactly the same rows.

In [9]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train_clean, X_test_clean, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train_clean:", X_train_clean.shape)
print("X_test_clean:", X_test_clean.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTrain target distribution:")
display(y_train.map(TARGET_NAMES).value_counts(normalize=True))

print("\nTest target distribution:")
display(y_test.map(TARGET_NAMES).value_counts(normalize=True))

X_train_clean: (3539, 36)
X_test_clean: (885, 36)
y_train: (3539,)
y_test: (885,)

Train target distribution:


TargetEncoded
Graduate    0.499294
Dropout     0.321277
Enrolled    0.179429
Name: proportion, dtype: float64


Test target distribution:


TargetEncoded
Graduate    0.499435
Dropout     0.320904
Enrolled    0.179661
Name: proportion, dtype: float64

## 7. Check existing split files

Before saving, compare the generated split with any existing
train/test CSV files.

Row order, columns, target values, and feature values must match, allowing only a small numerical serialization tolerance.

If either split differs, stop before writing outputs.

In [10]:
split_frames = {
    "train_clean.csv": (
        X_train_clean
        .assign(TargetEncoded=y_train.to_numpy())
        .reset_index(drop=True)
    ),
    "test_clean.csv": (
        X_test_clean
        .assign(TargetEncoded=y_test.to_numpy())
        .reset_index(drop=True)
    ),
}

for filename, expected_frame in split_frames.items():
    path = MODEL_INPUT_DIR / filename

    if not path.exists():
        print(f"{filename}: will be created.")
        continue

    existing_frame = pd.read_csv(path)

    try:
        pd.testing.assert_frame_equal(
            existing_frame,
            expected_frame,
            check_dtype=False,
            check_exact=False,
            rtol=1e-12,
            atol=1e-12,
        )
    except AssertionError as error:
        raise ValueError(
            f"{filename} differs from the generated split. "
            "No output files have been written. "
            "Review the source data and split settings."
        ) from error

    print(f"{filename}: existing split matches.")

train_clean.csv: existing split matches.
test_clean.csv: existing split matches.


## 8. Save shared model inputs

Output folder: `data/4_model_inputs/`

Saved files:

- `students_model_ready_from_powerbi.csv`: cleaned flat-table snapshot.
- `train_clean.csv`: training features and TargetEncoded.
- `test_clean.csv`: test features and TargetEncoded.
- `target_mapping.json`: target labels and encoded values.
- `split_metadata.json`: source, split settings, and feature definitions.

Existing matching train/test files are preserved without rewriting.

Model notebooks load train_clean.csv, test_clean.csv, and
split_metadata.json. They fit their own preprocessing Pipelines.

In [11]:
# Save the cleaned flat-table snapshot.
df.to_csv(
    MODEL_INPUT_DIR / "students_model_ready_from_powerbi.csv",
    index=False,
)

# Preserve existing matching split files.
for filename, frame in split_frames.items():
    path = MODEL_INPUT_DIR / filename

    if path.exists():
        print(f"Preserved: {filename}")
    else:
        frame.to_csv(path, index=False)
        print(f"Created: {filename}")

with open(
    MODEL_INPUT_DIR / "target_mapping.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        TARGET_MAPPING,
        file,
        indent=4,
    )

split_metadata = {
    "source_file": ML_READY_PATH.relative_to(
        PROJECT_ROOT
    ).as_posix(),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratified": True,
    "target_column": "TargetEncoded",
    "n_rows": int(df.shape[0]),
    "n_train_rows": int(X_train_clean.shape[0]),
    "n_test_rows": int(X_test_clean.shape[0]),
    "n_original_features": int(X.shape[1]),
    "feature_columns": X.columns.tolist(),
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "preprocessing_location": "model-specific pipelines",
}

with open(
    MODEL_INPUT_DIR / "split_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_metadata,
        file,
        indent=4,
    )

print("\nShared model inputs are ready:")
print(MODEL_INPUT_DIR)

Preserved: train_clean.csv
Preserved: test_clean.csv

Shared model inputs are ready:
D:\NU\semester_8\data_mining\project\phase I\data\4_model_inputs


## 9. Final check

All model notebooks load:

- `data/4_model_inputs/train_clean.csv`
- `data/4_model_inputs/test_clean.csv`
- `data/4_model_inputs/split_metadata.json`

Each split contains 36 original features and TargetEncoded.

Expected split sizes:

- Training: 3,539 rows.
- Test: 885 rows.

Models share the same students, target mapping, and feature definitions.
Fitted preprocessing is stored with each trained model.